# RAG Minecraft

## Configuration Gemini API

In [78]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers import MultiVectorRetriever
import pickle
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time
import re

In [79]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

### Config sources

In [80]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [81]:
scraper = cloudscraper.create_scraper()

In [82]:
def write_txt(file_name, paragraphs):
    # Create parent folder automatically if it does not exist
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

### Scraping Utils

#### Wikipedia

In [83]:
from langchain_core.documents import Document

def parse_wikipedia_sections(soup, source):

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_text = []

    def save_section():
        nonlocal current_text

        text = "\n".join(current_text).strip()

        # ignorer sections vides
        if not text:
            current_text = []
            return

        title_parts = [x for x in [h2, h3, h4] if x]
        section_title = " > ".join(title_parts)

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": source,
                    "section": section_title
                }
            )
        )

        current_text = []

    for tag in soup.find_all(["h2", "h3", "h4", "p", "li"]):

        txt = tag.get_text(" ", strip=True)

        if not txt:
            continue

        # arrêt avant les sections inutiles
        if tag.name == "h2" and any(
            x in txt
            for x in [
                "Références",
                "Notes et références",
                "Voir aussi",
                "Liens externes",
                "Accueil",
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h3" and any(
            x in txt
            for x in [
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h2":

            save_section()

            h2 = txt
            h3 = None
            h4 = None

        elif tag.name == "h3":

            save_section()

            h3 = txt
            h4 = None

        elif tag.name == "h4":

            save_section()

            h4 = txt

        else:
            current_text.append(txt)

    save_section()

    return docs

#### Fandom

In [84]:
def clean_wiki_text(text: str) -> str:

    # Guillemets typographiques → guillemets simples
    text = text.replace("\u00ab", '"').replace("\u00bb", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2026", "...")

    # Liens wiki [[Page|texte]] → texte, [[Page]] → Page
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)

    # Templates simples {{nom|val}} → val, {{nom}} → supprimé
    text = re.sub(r"\{\{[^}]*\|([^}]*)\}\}", r"\1", text)
    text = re.sub(r"\{\{[^}]*\}\}", "", text)

    # Balises HTML résiduelles
    text = re.sub(r"<[^>]+>", "", text)

    # Références [1], [note 2], etc.
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\[note \d+\]", "", text)

    # Marqueurs de version [Version Java 1.x]
    text = re.sub(r"\[Version [^\]]+\]", "", text)

    # Lignes qui ne contiennent que de la ponctuation ou des caractères spéciaux
    text = re.sub(r"^[^\w\s]{1,5}$", "", text, flags=re.MULTILINE)

    # Espaces multiples sur une même ligne
    text = re.sub(r"[ \t]+", " ", text)

    # Lignes vides multiples → une seule
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def remove_empty_sections(text: str) -> str:
    """
    Supprime les titres de section qui ne sont pas suivis de contenu.
    Ex : 'SECTION: Galerie\n\nSECTION: Combat\n' → garde seulement Combat s'il a du contenu.
    """
    lines = text.split("\n")
    result = []
    i = 0

    while i < len(lines):
        line = lines[i]

        if line.startswith("SECTION:") or line.startswith("SOUS-SECTION:"):
            # Chercher si la section a du contenu
            j = i + 1
            # Sauter les lignes vides juste après le titre
            while j < len(lines) and lines[j].strip() == "":
                j += 1

            # S'il y a du contenu non-vide et que ce n'est pas une autre section
            if j < len(lines) and not (
                lines[j].startswith("SECTION:") or lines[j].startswith("SOUS-SECTION:")
            ):
                result.append(line)
            # Sinon on saute le titre de section vide
        else:
            result.append(line)

        i += 1

    return "\n".join(result)

def remove_noise_lines(text: str) -> str:
    """
    Supprime les lignes qui ne contiennent pas assez d'information :
    - moins de 20 caractères sauf si c'est un titre de section
    - lignes qui ressemblent à des noms de fichier (image, .png, .jpg)
    - lignes qui ne sont que des chiffres ou caractères spéciaux
    """
    lines = text.split("\n")
    result = []

    for line in lines:
        stripped = line.strip()

        # Toujours garder les titres de section
        if stripped.startswith("SECTION:") or stripped.startswith("SOUS-SECTION:"):
            result.append(line)
            continue

        # Supprimer les noms de fichiers image
        if re.match(r"(?i).*\.(png|jpg|jpeg|gif|svg|webp|ogg|mp3|wav)$", stripped):
            continue

        # Supprimer les lignes trop courtes (bruit)
        if len(stripped) < 20 and stripped != "":
            continue

        # Supprimer les lignes qui ne sont que des chiffres/ponctuation
        if re.match(r"^[\d\s.,;:!?()\[\]«»\"'/-]+$", stripped):
            continue

        if stripped.startswith("v d m"):
            continue

        if stripped.startswith("↑"):
            continue

        if "google.fr/search" in stripped:
            continue

        if "Version Bedrock" in stripped and len(stripped) > 100:
            continue

        if "Historique des versions" in stripped and len(stripped) > 100:
            continue

        result.append(line)

    return "\n".join(result)

STOP_HEADINGS = {
    "références", "notes et références", "notes diverses",
    "voir aussi", "liens externes", "annexes",
    "galerie", "historique", "succès",
    "créatures des autres jeux", "créatures de minecraft earth",
    "créatures de minecraft dungeons", "créatures prévues",
    "créatures inutilisées", "créatures supprimées",
    "créatures non implémentées", "créatures de la version éducation",
    "tutoriels",
}

def extract_list(tag) -> list:
    items = []
    for li in tag.find_all("li", recursive=False):
        text_parts = []
        for node in li.children:
            if not hasattr(node, "name"):
                text_parts.append(str(node))
            elif node.name not in ["ul", "ol"]:
                text_parts.append(node.get_text(" ", strip=True))
        text = clean_wiki_text(" ".join(text_parts))
        if text and len(text) >= 15:
            items.append(f"- {text}")
        for sub in li.find_all(["ul", "ol"], recursive=False):
            for sub_item in extract_list(sub):
                items.append(f"  {sub_item}")
    return items

def get_cell_name(td):
    td = BeautifulSoup(str(td), "html.parser")

    # Remplacer les liens par leur title
    for a in td.find_all("a"):
        title = a.get("title", "").strip()
        if title and "modifier" not in title.lower():
            a.replace_with(title)

    # Remplacer les images par leur alt
    for img in td.find_all("img"):
        alt = img.get("alt", "").strip()
        if alt:
            img.replace_with(alt)

    return td.get_text(" ", strip=True)

def extract_table(tag) -> list:
    rows = []

    # Récupérer les en-têtes
    headers = []
    header_row = tag.find("tr")
    if header_row:
        headers = [
            clean_wiki_text(th.get_text(" ", strip=True))
            for th in header_row.find_all("th")
            if len(th.get_text(strip=True)) >= 2
        ]

    for tr in tag.find_all("tr"):
        cells = []
        for td in tr.find_all("td"):
            cell_text = clean_wiki_text(get_cell_name(td))
            if cell_text and len(cell_text) >= 2:
                cells.append(cell_text)

        if not cells:
            continue

        if headers and len(headers) == len(cells):
            parts = [f"{h} : {c}" for h, c in zip(headers, cells)]
            rows.append("- " + " | ".join(parts))
        else:
            rows.append("- " + " : ".join(cells))

    return rows


### Wikipedia

In [85]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(
        url,
        params=params,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")
    docs = parse_wikipedia_sections(soup, f"wikipedia:{title}" )
    
    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        } 
        for doc in docs
    ]

    write_txt(
        f"files/wikipedia.txt",
        docs
    )
    return docs


### Fandom

In [93]:
def scrape_fandom(url: str):

    headers_req = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "fr-FR,fr;q=0.9",
        "Referer": "https://www.google.com/"
    }

    r = scraper.get(url, headers=headers_req)
    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return []

    # ==========================
    # CLEAN STATIC BLOCKS
    # ==========================
    selectors_to_remove = [
        "#toc",
        ".mw-references-wrap",
        "ol.references",
        ".navbox",
        ".vertical-navbox",
        ".portable-infobox",
        ".catlinks",
        ".mw-editsection",
        ".reference"
    ]

    for selector in selectors_to_remove:
        for element in content.select(selector):
            element.decompose()

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_content = []

    def save_section():
        nonlocal current_content

        text = "\n".join(current_content).strip()

        text = remove_empty_sections(text)
        text = remove_noise_lines(text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        if not text:
            current_content = []
            return

        section_parts = [x for x in [h2, h3, h4] if x]

        section_title = (
            "Introduction"
            if not section_parts
            else " > ".join(section_parts)
        )

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": url,
                    "section": section_title
                }
            )
        )

        current_content = []

    # ==========================
    # MAIN LOOP
    # ==========================
    for tag in content.children:

        if not getattr(tag, "name", None):
            continue

        # ==========================
        # HEADINGS — h1 inclus pour FAQ
        # ==========================
        heading_tag = None

        if tag.name in ["h1", "h2", "h3", "h4"]:
            heading_tag = tag
        elif tag.name == "div" and "mw-heading" in tag.get("class", []):
            heading_tag = tag.find(["h1", "h2", "h3", "h4"], recursive=False)

        if heading_tag:
            raw_heading = heading_tag.get_text(" ", strip=True)
            heading = clean_wiki_text(raw_heading)
            heading = re.sub(r"\[.*?\]", "", heading).strip()

            if not heading:
                continue

            # ==========================
            # STOP SECTIONS
            # ==========================
            if heading.lower() in STOP_HEADINGS:
                save_section()
                break

            # ==========================
            # FAQ DETECTION
            # ==========================
            b_texts = [b.get_text(strip=True).lower() for b in heading_tag.find_all("b")]
            is_faq = any(t == "q:" for t in b_texts)

            if is_faq:
                # Sauvegarder la section précédente (Q/R précédente)
                save_section()

                # Extraire le texte de la question sans le préfixe "Q:"
                question_text = clean_wiki_text(heading)
                question_text = re.sub(r"^Q\s*:\s*", "", question_text, flags=re.IGNORECASE).strip()

                # Chaque question devient sa propre section
                h2 = f"FAQ > {question_text}"
                h3 = None
                h4 = None

                # Ajouter la question en début de contenu
                current_content.append(f"Q: {question_text}")
                continue

            # ==========================
            # SECTION NORMALE
            # ==========================
            save_section()

            if heading_tag.name in ["h1", "h2"]:
                h2 = heading
                h3 = None
                h4 = None
            elif heading_tag.name == "h3":
                h3 = heading
                h4 = None
            elif heading_tag.name == "h4":
                h4 = heading

            continue

        # ==========================
        # PARAGRAPHS
        # ==========================
        if tag.name == "p":
            text = clean_wiki_text(tag.get_text(" ", strip=True))
            if not text or len(text) < 10:
                continue
            current_content.append(text)

        # ==========================
        # DL/DD — réponses FAQ et définitions
        # ==========================
        elif tag.name == "dl":
            for dd in tag.find_all("dd", recursive=True):
                raw = dd.get_text(" ", strip=True)
                # Supprimer le préfixe "R:" s'il existe
                raw = re.sub(r"^R\s*:\s*", "", raw, flags=re.IGNORECASE).strip()
                text = clean_wiki_text(raw)
                if not text or len(text) < 10:
                    continue
                current_content.append(text)

        # ==========================
        # LISTS
        # ==========================
        elif tag.name in ["ul", "ol"]:
            current_content.extend(extract_list(tag))

        # ==========================
        # TABLES
        # ==========================
        elif tag.name == "table":
            current_content.extend(extract_table(tag))

    save_section()

    page_name = unquote(url.split("/wiki/")[-1])

    print(f"FANDOM OK: {page_name} ({len(docs)} sections)")

    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        }
        for doc in docs
    ]

    write_txt(f"files/{page_name}.txt", docs)

    return docs

### Build dataset

In [95]:
docs = []

# Wikipedia
for page in WIKI_PAGES:
    try:
        docs.extend(scrape_wikipedia(page))
        print("WIKI OK:", page)
    except Exception as e:
        print("WIKI ERROR:", page, e)

# Fandom
for url in FANDOM_URLS:
    try:
        doc = scrape_fandom(url)
        if doc:
            docs.extend(doc)
            print("FANDOM OK:", url)
    except Exception as e:
        print("FANDOM ERROR:", url, e)

print("\nTOTAL DOCS:", len(docs))
for i, x in enumerate(docs):
    print(x["section"], ":", len(x["page_content"]))

WIKI OK: Minecraft
FANDOM OK: Survie (10 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Survie
FANDOM OK: Créatif (6 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif
FANDOM OK: Hardcore (3 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Hardcore
FANDOM OK: Fabrication (4 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Fabrication
FANDOM OK: Cuisson (7 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cuisson
FANDOM OK: Alchimie (9 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Alchimie
FANDOM OK: La_Foire_aux_Questions (14 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions
FANDOM OK: Tutoriels/Choses_à_ne_PAS_faire (24 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire
FANDOM OK: Tutoriels/Survie_dans_le_Nether (17 sections)
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether
FANDOM OK: Tutoriels/L'End_et_l'Ender_D

Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

# i = 0
# while i < 3:
#     print("Chunk", i)
#     print(split_docs[i])
#     i +=1

print("CHUNKS:", len(split_docs))

CHUNKS: 0


Enrchich

In [ ]:
def enrich_with_source(docs):
    enriched = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        # Extraire le nom de la page depuis l'URL comme "pseudo-titre"
        if "wiki/" in src:
            page_title = src.split("wiki/")[-1].replace("_", " ")
        elif "wikipedia:" in src:
            page_title = src.split("wikipedia:")[-1].replace("_", " ")
        else:
            page_title = src

        # Enrichissement du chunk : on ajoute le contexte AVANT le texte
        # → la similarité vectorielle bénéficie du titre + type de source
        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n"
            f"TOPIC: {page_title}\n\n"   # ← nouveauté
            f"{d.page_content}"
        )
        enriched.append(d)
    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

ModuleNotFoundError: No module named 'langchain_ollama'

#### Store data using chroma

In [ ]:
import shutil
import os
path = "./chroma_minecraft_db"

if os.path.exists(path):
    try:
        shutil.rmtree(path)
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")

In [ ]:
# --- ÉTAPE 1 : Générer un résumé par chunk ---

summarize_chain = (
    PromptTemplate.from_template(
        """
Résume cette page Minecraft pour un moteur de recherche sémantique.

Mentionne explicitement :
- les principaux concepts ;
- les créatures ;
- les objets ;
- les structures ;
- les biomes ;
- les mécaniques de jeu.

Le résumé doit faire entre 100 et 200 mots.

{doc}
"""
    )
    | llm
    | StrOutputParser()
)


summaries = []
for doc in docs:
    doc.
    # summaries.append(summary)
    write_txt("summaries/")

In [ ]:


# --- ÉTAPE 2 : Associer chaque résumé à son chunk original ---
# On crée un ID unique par chunk pour faire le lien
# résumé (dans Chroma) <-> chunk original (dans le docstore)
import uuid
doc_ids = [str(uuid.uuid4()) for _ in split_docs]

# Les résumés sont les documents indexés dans Chroma
# On colle l'ID dans les metadata pour retrouver l'original
summary_docs = [
    Document(
        page_content=summaries[i],
        metadata={"doc_id": doc_ids[i]}  # clé de liaison
    )
    for i in range(len(split_docs))
]

# --- ÉTAPE 3 : Construire le retriever ---
# InMemoryByteStore = le "tiroir" qui stocke les chunks originaux
# Le MultiVectorRetriever fait le pont :
#   1. cherche dans Chroma (résumés)
#   2. récupère l'ID du résumé trouvé
#   3. va chercher le chunk original dans le docstore
#   4. retourne le chunk original au LLM


vectorstore = Chroma(
    persist_directory="./chroma_minecraft_multivec",
    embedding_function=gemini_embeddings
)
vectorstore.add_documents(summary_docs)

store = InMemoryByteStore()  # stockage des chunks originaux en mémoire

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,   # Chroma contient les résumés
    byte_store=store,          # le store contient les chunks bruts
    id_key="doc_id",           # la clé qui fait le lien
)

# On peuple le docstore avec les chunks originaux
retriever.docstore.mset(list(zip(doc_ids, split_docs)))

In [ ]:
# Check the number of chunks generated after text splitting.
print("split_docs:", len(split_docs))

# Check the number of documents actually stored in the Chroma vector database.
# If this number is equal to len(split_docs), the database was built without missing or duplicated chunks.
print("chroma:", vectorstore._collection.count())

#### Create a retriever using Chroma

In [ ]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

## Generator

#### Create prompt templates

In [ ]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

#### Create a stuff documents chain

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [ ]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        nouvelle_requete = str(llm_rewrite.invoke(rewrite_prompt).content)
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [ ]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

In [ ]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft? Décris les."))

In [ ]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft, et décris les tous."))

In [ ]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)